# Part 4: A baseline, and feature extraction

Before you fine-tune anything, find out what is easy. A classical baseline sets the bar. **Feature extraction** uses a pretrained model without changing it: it turns each text into a vector, and a simple classifier learns from those vectors.

In [ ]:
import warnings

from transformers.utils import logging

logging.set_verbosity_error()  # hide the long model-loading reports
warnings.filterwarnings("ignore")

from review_data import as_lists, load_reviews, small_sample

reviews = load_reviews()
val_texts, val_labels = as_lists(reviews["validation"])

texts, labels = small_sample(reviews["train"], 200, seed=42)

## The baseline: word counts and logistic regression

In [ ]:
from baselines import tfidf_baseline

print(tfidf_baseline(texts, labels, val_texts, val_labels))

## Feature extraction: a frozen DistilBERT

Nothing in the pretrained model is trained here. Each text becomes one vector of 768 numbers (the mean of its token vectors, ignoring padding).

In [ ]:
from embed import embed_texts

vectors = embed_texts(texts[:4])
print(vectors.shape)

In [ ]:
from sklearn.linear_model import LogisticRegression

from baselines import scores

classifier = LogisticRegression(max_iter=2000).fit(embed_texts(texts), labels)
print(scores(val_labels, classifier.predict(embed_texts(val_texts))))

With 200 labelled examples the frozen features are about 11 points better than word counts. The pretrained model already knows which words and phrases go together.

## Try it

1. Repeat with 50 examples and with 500. Which method gains more from extra data?
2. Change `C` in the logistic regression. Does it help?
3. Why do we fit the classifier on the training sample only?